# Scenario: Popular Post Analysis

**Business requirement**:

The analytics department wants a dataset containing popular posts that meet specific engagement criteria.

**Transformation requirements**:
1. Extract id, title, tags, views, userId, reactions.likes, and reactions.dislikes.
2. Only retain posts where views is greater than 200.
3. Only retain posts where likes is greater than dislikes.
4. Convert all tags into lowercase.
5. Calculate engagement_rate.
6. Create a popularity_status column:
   - Engagement rate >= 50%: Popular
   - Engagement rate >= 20%: Moderate
   - Otherwise: Low
7. Export the result into PostgreSQL database.

**Expected output schema**
| Column | Data type |
-------- | --------- |
| post_id       | Integer
| title    | String
| tags     | Array/String
| views    | Integer
| user_id  | Integer
| likes    | Integer
| dislikes | Integer
| engagement_rate | Float
| popularity_status | String

Note: Define engagement rate as (likes + dislikes) / views × 100. Handle zero or missing views explicitly.

In [88]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

In [89]:
load_dotenv()

True

## Extract Data

In [90]:
posts_data = pd.read_json('./data/posts.json')['posts']

df = pd.json_normalize(
  posts_data,
  # record_path="tags",
  meta=["id", "title", "tags", "views", "userId", ["reactions", "likes"], ["reactions", "dislikes"]]
  )

# df['posts']['title'] = df['posts']['title'].str.lower()
df = df.rename(columns={
  "id": "post_id",
  "userId": "user_id",
  "reactions.likes": 'likes',
  "reactions.dislikes": 'dislikes'
})
# print(df)
df = df[['post_id', 'title', 'tags', 'views', 'user_id', 'likes', 'dislikes']]

In [91]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   post_id   251 non-null    int64 
 1   title     251 non-null    str   
 2   tags      251 non-null    object
 3   views     251 non-null    int64 
 4   user_id   251 non-null    int64 
 5   likes     251 non-null    int64 
 6   dislikes  251 non-null    int64 
dtypes: int64(5), object(1), str(1)
memory usage: 13.9+ KB


In [92]:
df.describe()

,post_id,views,user_id,likes,dislikes
count,251.000000,251.000000,251.000000,251.000000,251.000000
mean,126.000000,2390.848606,107.330677,616.011952,23.976096
std,72.601653,1477.309712,59.025132,434.175305,14.613946
min,1.000000,8.000000,1.000000,1.000000,0.000000
25%,63.500000,1157.500000,58.000000,223.500000,11.000000
50%,126.000000,2308.000000,108.000000,552.000000,23.000000
75%,188.500000,3627.000000,155.500000,980.500000,37.000000
max,251.000000,4993.000000,208.000000,1489.000000,50.000000


In [93]:
df.sample(5)

,post_id,title,tags,views,user_id,likes,dislikes
108,109,"How vulgar, this hankering after","[mystery, classic, history]",2380,23,815,36
49,50,It was a concerning development that he couldn...,"[french, fiction, love]",1455,15,1136,49
10,11,It wasn't quite yet time to panic.,"[mystery, american, history]",984,43,453,8
246,247,The cozy bookstore was a haven for book lovers,"[bookstore, books, cozy]",2054,132,1139,11
156,157,The old man told tales of his youth,"[stories, memory, emotion]",3142,92,610,22


In [94]:
df.isnull().sum()

post_id     0
title       0
tags        0
views       0
user_id     0
likes       0
dislikes    0
dtype: int64

In [95]:
df.isna().sum()

post_id     0
title       0
tags        0
views       0
user_id     0
likes       0
dislikes    0
dtype: int64

## Transform Data

In [96]:
df_filtered = df[
  (df['views'] > 200) &
  (df['likes'] > df['dislikes'])
]

In [97]:
# tags to all lowercase
df_filtered['tags'].apply(lambda tags: [t.lower() for t in tags])

0         [history, american, crime]
1         [french, fiction, english]
2         [magical, history, french]
3       [mystery, english, american]
4             [crime, mystery, love]
                   ...              
246         [bookstore, books, cozy]
247           [city, skyline, night]
248         [temple, ancient, faith]
249      [village, square, peaceful]
250    [waterfall, majestic, nature]
Name: tags, Length: 236, dtype: object

In [98]:
# engagement_rate column addition
engagement_rate = ((df_filtered['likes'] + df_filtered['dislikes']) / df_filtered['views']) * 100
df_filtered['engagement_rate'] = engagement_rate

In [99]:
# popularity_status column addition
conditions = [
  (df_filtered['engagement_rate'] >= 50),
  (df_filtered['engagement_rate'] >= 20),
  (df_filtered['engagement_rate'] < 20)
]

choices = ['Popular', 'Moderate', 'Low']

df_filtered['popularity_status'] = np.select(conditions, choices, default='N/A')

In [100]:
df_filtered.info()

<class 'pandas.DataFrame'>
Index: 236 entries, 0 to 250
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   post_id            236 non-null    int64  
 1   title              236 non-null    str    
 2   tags               236 non-null    object 
 3   views              236 non-null    int64  
 4   user_id            236 non-null    int64  
 5   likes              236 non-null    int64  
 6   dislikes           236 non-null    int64  
 7   engagement_rate    236 non-null    float64
 8   popularity_status  236 non-null    str    
dtypes: float64(1), int64(5), object(1), str(2)
memory usage: 18.4+ KB


In [101]:
df_filtered.describe()

,post_id,views,user_id,likes,dislikes,engagement_rate
count,236.000000,236.000000,236.000000,236.000000,236.000000,236.000000
mean,126.809322,2536.538136,107.241525,651.330508,24.500000,34.653721
std,72.054956,1401.697030,59.522684,423.519068,14.534625,24.770836
min,1.000000,220.000000,1.000000,15.000000,0.000000,0.473293
25%,65.750000,1336.250000,57.750000,271.750000,12.000000,15.841363
50%,126.500000,2461.500000,107.500000,591.000000,24.000000,28.068342
75%,188.250000,3736.750000,156.250000,1007.000000,37.000000,50.073234
max,251.000000,4993.000000,208.000000,1489.000000,50.000000,101.002506


In [102]:
df_filtered.isna().sum()

post_id              0
title                0
tags                 0
views                0
user_id              0
likes                0
dislikes             0
engagement_rate      0
popularity_status    0
dtype: int64

In [103]:
df_filtered.isnull().sum()

post_id              0
title                0
tags                 0
views                0
user_id              0
likes                0
dislikes             0
engagement_rate      0
popularity_status    0
dtype: int64

In [104]:
df_filtered.sample(5)

,post_id,title,tags,views,user_id,likes,dislikes,engagement_rate,popularity_status
228,229,The charming cottage nestled in a sun-dappled ...,"[cottage, charming, peace]",3264,138,352,30,11.703431,Low
72,73,It was difficult to explain to them,"[fiction, classic, magical]",3462,155,389,37,12.305026,Low
190,191,The mountain trail was challenging,"[mountain, trail, challenge]",4432,134,62,15,1.737365,Low
110,111,"Act, implores the Ghost of Future Regret.","[crime, love]",1481,112,623,39,44.699527,Moderate
178,179,The old theater was a relic of a glamorous past,"[theater, history, culture]",4973,83,1128,1,22.702594,Moderate


## Load Data

In [105]:
db = os.environ["DATABASE_URL"]
engine = create_engine(db)

df_filtered.to_sql(
  name="popular_posts",
  con=engine,
  if_exists="replace",
  index=False
)

236

In [106]:
result = pd.read_sql(
  "SELECT * FROM popular_posts",
  con=engine
)

print(result)

     post_id                                              title  \
0          1                   His mother had always taught him   
1          2           He was an expert but not in a discipline   
2          3  Dave watched as the forest burned up on the hill.   
3          4                     All he wanted was a candy bar.   
4          5             Hopes and dreams were dashed that day.   
..       ...                                                ...   
231      247     The cozy bookstore was a haven for book lovers   
232      248    The vibrant city skyline glittered in the night   
233      249  The ancient temple stood as a testament to hum...   
234      250  The peaceful village square was a gathering pl...   
235      251  The majestic waterfall thundered down the moun...   

                            tags  views  user_id  likes  dislikes  \
0       {history,american,crime}    305      121    192        25   
1       {french,fiction,english}   4884       91    859  